# Volta River Basin Flood-Susceptibility Analysis
GitHub-ready reconstruction of the final redesigned RF/XGBoost workflow.

**Reproducibility note:** exact historical seed, model hyperparameters and five-class thresholds were not recoverable; configurable defaults are clearly marked in the code.


In [ ]:
# Volta Flood Susceptibility — final ML reconstruction
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, joblib, warnings
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from xgboost import XGBClassifier
import shap
warnings.filterwarnings('ignore')

BASE=Path('/content/drive/MyDrive/VOLTA_FLOOD_FINAL_CORRECTED_RESULTS')
DATA=BASE/'DATA'; FIG=BASE/'FIGURES'; TAB=BASE/'TABLES'; MOD=BASE/'MODELS'
for p in [FIG,TAB,MOD]: p.mkdir(parents=True,exist_ok=True)
PRED_TIF=DATA/'Volta_15_Predictors_1km.tif'
WATER_TIF=DATA/'Volta_Permanent_Water_Mask_1km.tif'
REF_CSV=DATA/'Volta_Final_Reference_Samples.csv'
SEED=42
P=['elevation','distance_to_river','drainage_density','landcover','clay','NDVI','HAND','annual_rainfall','max30day_rainfall','relative_elevation','slope','Rx5day','Rx1day','TWI','log_flow_accumulation']

# Reference CSV: longitude, latitude, label (1 flood; 0 non-flood).
df=pd.read_csv(REF_CSV)
assert {'longitude','latitude','label'}.issubset(df.columns)
df['block_x']=np.floor(df.longitude/.5).astype(int); df['block_y']=np.floor(df.latitude/.5).astype(int)
df['block_id']=df.block_x.astype(str)+'_'+df.block_y.astype(str)

# Extract predictors if not already present.
if not set(P).issubset(df.columns):
    with rasterio.open(PRED_TIF) as src:
        vals=np.array(list(src.sample(list(zip(df.longitude,df.latitude)))))
    if vals.shape[1]!=15: raise ValueError('Predictor raster must contain 15 bands.')
    for i,c in enumerate(P): df[c]=vals[:,i]

# Spatial blocks: target design = 127 train groups + 43 independent holdout groups.
blocks=np.array(sorted(df.block_id.unique())); np.random.default_rng(SEED).shuffle(blocks)
if len(blocks)<170: raise ValueError(f'Need >=170 blocks; found {len(blocks)}')
train_blocks=set(blocks[:127]); test_blocks=set(blocks[127:170])
train=df[df.block_id.isin(train_blocks)].replace([np.inf,-np.inf],np.nan).dropna(subset=P+['label'])
test=df[df.block_id.isin(test_blocks)].replace([np.inf,-np.inf],np.nan).dropna(subset=P+['label'])
Xtr,ytr=train[P],train.label.astype(int); Xte,yte=test[P],test.label.astype(int)
print('groups:',train.block_id.nunique(),test.block_id.nunique(),'samples:',len(train),len(test))

# Reproducible defaults. Exact historical hyperparameters/seed were not recoverable.
rf=RandomForestClassifier(n_estimators=500,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=-1)
xgb=XGBClassifier(n_estimators=500,max_depth=6,learning_rate=.05,subsample=.8,colsample_bytree=.8,eval_metric='logloss',random_state=SEED,n_jobs=-1)
rf.fit(Xtr,ytr); xgb.fit(Xtr,ytr)

def eval_model(name,m):
    pred=m.predict(Xte); prob=m.predict_proba(Xte)[:,1]; tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
    return {'Model':name,'Accuracy':accuracy_score(yte,pred),'Balanced Accuracy':balanced_accuracy_score(yte,pred),
    'Precision':precision_score(yte,pred),'Sensitivity':recall_score(yte,pred),'Specificity':tn/(tn+fp),
    'F1':f1_score(yte,pred),'ROC-AUC':roc_auc_score(yte,prob),'Average Precision':average_precision_score(yte,prob)},pred,prob
rm,rpred,rprob=eval_model('Random Forest',rf); xm,xpred,xprob=eval_model('XGBoost',xgb)
metrics=pd.DataFrame([rm,xm]); display(metrics); metrics.to_csv(TAB/'Model_Performance.csv',index=False)

print('Reported final-study benchmark for verification:')
print('RF: Accuracy=.9526, Balanced=.9506, Precision=.9746, Sensitivity=.9220, Specificity=.9792, F1=.9476, AUC=.9923, AP=.9906')
print('XGBoost: Accuracy=.9495, AUC=.9901')

# RF importance
imp=pd.DataFrame({'Predictor':P,'Importance':rf.feature_importances_}).sort_values('Importance',ascending=False)
imp.to_csv(TAB/'RF_Predictor_Importance.csv',index=False)
plt.figure(figsize=(8,7)); plt.barh(imp.Predictor[::-1],imp.Importance[::-1]); plt.xlabel('Random Forest importance'); plt.tight_layout()
plt.savefig(FIG/'Figure_RF_Predictor_Importance.png',dpi=600,bbox_inches='tight'); plt.show()

# SHAP
sample=Xte.sample(min(3000,len(Xte)),random_state=SEED)
explainer=shap.TreeExplainer(xgb); sv=explainer.shap_values(sample)
shap.summary_plot(sv,sample,show=False); plt.tight_layout(); plt.savefig(FIG/'Figure_SHAP_Summary.png',dpi=600,bbox_inches='tight'); plt.show()

# ROC / PR
plt.figure(figsize=(7,6))
for n,pb in [('Random Forest',rprob),('XGBoost',xprob)]:
    fpr,tpr,_=roc_curve(yte,pb); plt.plot(fpr,tpr,label=f'{n} (AUC={roc_auc_score(yte,pb):.4f})')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend(); plt.tight_layout()
plt.savefig(FIG/'Figure_Spatial_Holdout_ROC.png',dpi=600,bbox_inches='tight'); plt.show()

# Basin-wide RF probability
with rasterio.open(PRED_TIF) as src:
    a=src.read().astype('float32'); profile=src.profile.copy(); nodata=src.nodata
flat=a.reshape(a.shape[0],-1).T; valid=np.all(np.isfinite(flat),axis=1)
if nodata is not None: valid &= np.all(flat!=nodata,axis=1)
pf=np.full(len(flat),np.nan,dtype='float32')
pf[valid]=rf.predict_proba(pd.DataFrame(flat[valid],columns=P))[:,1]
prob=pf.reshape(a.shape[1],a.shape[2])
if WATER_TIF.exists():
    with rasterio.open(WATER_TIF) as w: water=w.read(1)
    prob[water==1]=np.nan
profile.update(count=1,dtype='float32',nodata=-9999,compress='lzw')
with rasterio.open(BASE/'Final_Flood_Susceptibility_Probability.tif','w',**profile) as dst:
    dst.write(np.where(np.isfinite(prob),prob,-9999).astype('float32'),1)

# Exact historical five-class thresholds were not recoverable.
# Replace these defaults if the original thresholds are recovered.
BREAKS=[0,.2,.4,.6,.8,1.000001]
cl=np.zeros(prob.shape,dtype='uint8')
for i in range(5): cl[(prob>=BREAKS[i])&(prob<BREAKS[i+1])]=i+1
cl[~np.isfinite(prob)]=0
cp=profile.copy(); cp.update(dtype='uint8',nodata=0)
with rasterio.open(BASE/'Final_Flood_Susceptibility_Classes.tif','w',**cp) as dst: dst.write(cl,1)

joblib.dump(rf,MOD/'Random_Forest.joblib'); joblib.dump(xgb,MOD/'XGBoost.joblib')
with open(MOD/'analysis_metadata.json','w') as f:
    json.dump({'predictors':P,'scale_m':1000,'block_deg':.5,'train_groups':127,'holdout_groups':43,
    'reconstruction_seed':SEED,'reconstruction_class_breaks':BREAKS,
    'warning':'Seed, model hyperparameters and class thresholds are reconstructed defaults, not verified historical settings.'},f,indent=2)
print('DONE:',BASE)
